In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
class BiomassNet(nn.Module):
    def __init__(self, input_dim, hid_dim1=64, hid_dim2=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hid_dim1) # Input to Hidden Dimension 1
        self.fc2 = nn.Linear(hid_dim1, hid_dim2) # Hidden Dimension 1-2
        self.out = nn.Linear(hid_dim2, 1) # Hidden Dimension 2 to Biomass Prediction
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

In [ ]:
df = pd.read_csv('train_biomass.csv')

# Drop useless column
df = df.drop(columns=['Unnamed: 0'])

# Split features / target
X = df.drop(['TT_DW_CRM', 'SPCD'], axis=1)
y = df['TT_DW_CRM']

# Scale numerical columns
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.float32).view(-1,1)

In [ ]:
def RMSELoss(pred, target):
    return torch.sqrt(nn.MSELoss()(pred, target))

In [ ]:
model = BiomassNet(input_dim=X_tensor.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 200

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()
    preds = model(X_tensor)
    loss = RMSELoss(preds, y_tensor)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs}, RMSE = {loss.item():.4f}")

Epoch 20/200, RMSE = 14456.4766
Epoch 40/200, RMSE = 14456.0615
Epoch 60/200, RMSE = 14455.3340
Epoch 80/200, RMSE = 14454.1045
Epoch 100/200, RMSE = 14452.1641
Epoch 120/200, RMSE = 14449.2627
Epoch 140/200, RMSE = 14445.1455
Epoch 160/200, RMSE = 14439.4766
Epoch 180/200, RMSE = 14431.8838
Epoch 200/200, RMSE = 14421.9990


In [ ]:
df1 = pd.read_csv('test_biomass.csv')

# Drop useless column
df1 = df1.drop(columns=['Unnamed: 0'])

# Split features / target
X_test = df1.drop(['TT_DW_CRM', 'SPCD'], axis=1)
y_test = df1['TT_DW_CRM']

# Scale numerical columns
scaler1 = StandardScaler()
X_scaled = scaler1.fit_transform(X_test)

# Convert to tensors
X_test_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y.values, dtype=torch.float32).view(-1,1)

In [ ]:
model.eval()

with torch.no_grad():
    test_preds = model(X_test_tensor)
    test_rmse = RMSELoss(test_preds, y_test_tensor)

print("Test RMSE:", test_rmse.item())

Test RMSE: 14451.3779296875
